# Games Backtest (2026 default)

Backtest variant of `games_today_tomorrow.ipynb`:

- Defaults to **completed 2026 games** from `data/games_2026.parquet`.
- Keeps V6-style scoring logic, but changes data selection upstream.
- If Kalshi historical implied columns are missing, scoring still runs and Kalshi-based metrics are omitted (set to NaN).

Run top-to-bottom after refreshing Parquet files.

In [9]:
# Backtest parameters (edit these)
SEASON = 2026
START_DATE = "2026-03-25"  # inclusive, YYYY-MM-DD
END_DATE = None             # inclusive; None = today

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

print({
    "SEASON": SEASON,
    "START_DATE": START_DATE,
    "END_DATE": END_DATE or "today",
    "FINAL_STATES": sorted(FINAL_STATES),
})

{'SEASON': 2026, 'START_DATE': '2026-03-25', 'END_DATE': 'today', 'FINAL_STATES': ['Completed Early', 'Final', 'Game Over']}


In [10]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run: python -m app.main live --season {SEASON}")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

df_bt = df[df["detailed_state"].isin(FINAL_STATES)].copy()

if "home_win" in df_bt.columns:
    # ensure final training label is present for backtest
    df_bt = df_bt[df_bt["home_win"].notna()].copy()

start_ts = pd.Timestamp(START_DATE).normalize()
end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE).normalize()
if end_ts < start_ts:
    raise ValueError(f"END_DATE ({end_ts.date()}) must be >= START_DATE ({start_ts.date()})")

sub = df_bt[df_bt["game_date"].between(start_ts, end_ts)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)}  completed={len(df_bt)}")
if len(df_bt):
    print(f"Date range completed: {df_bt['game_date'].min().date()} .. {df_bt['game_date'].max().date()}")
print(f"Backtest window: {start_ts.date()} .. {end_ts.date()}  rows={len(sub)}")


Using: C:\Users\Austin\baseball\mlb-model\data\games_2026.parquet
Rows total=2796  completed=600
Date range completed: 2026-03-01 .. 2026-04-15
Backtest window: 2026-03-25 .. 2026-04-15  rows=282


In [11]:
# Core thresholds from your mismatch workflow
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2

USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

favorites = s.copy().sort_values(["game_date", "home_team_name"]).reset_index(drop=True)
favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = np.where(
    favorites["_mismatch_side"].eq("home_favored"),
    favorites["home_team_name"],
    favorites["away_team_name"],
)

print("Favorites frame rows:", len(favorites))
display(favorites.tail(25))

Favorites frame rows: 282


,game_pk,game_date,detailed_state,home_team_name,away_team_name,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher,away_probable_pitcher,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,sp_kbb_diff,sp_xfip_diff,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_fip,away_pen_season_fip,home_pen_season_era,away_pen_season_era,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,_mismatch_side,core_matches,favored_team
257,823964,2026-04-13,Final,Los Angeles Dodgers,New York Mets,119,121,4.0,0.0,Justin Wrobleski,David Peterson,680736,656849,L,L,119.549194,89.162585,1.587302,11.340206,3.276471,3.303390,1.0,True,2026,2026,-9.752905,-0.026919,30.386609,129.650787,97.014555,32.636233,-2.631579,14.893617,3.877778,2.028571,3.289189,3.247887,2.675676,1.711268,3.850000,2.977551,3.907895,1.377551,0.560811,-0.270336,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,New York Mets
258,823725,2026-04-13,Final,Minnesota Twins,Boston Red Sox,142,111,13.0,6.0,Bailey Ober,Garrett Crochet,641927,676979,R,L,104.427220,96.580912,10.588235,16.304348,3.913559,4.573684,1.0,True,2026,2026,-5.716113,-0.660125,7.846307,104.912821,94.995323,9.917498,7.142857,24.000000,2.789655,3.100000,3.824138,4.240496,3.956897,3.123967,4.045652,3.841176,4.402174,3.494118,0.221514,-0.399319,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Boston Red Sox
259,823563,2026-04-13,Final,New York Yankees,Los Angeles Angels,147,108,11.0,10.0,Will Warren,Yusei Kikuchi,701542,579328,R,L,98.007513,103.285938,17.500000,10.112360,3.481818,4.822222,1.0,True,2026,2026,7.387640,-1.340404,-5.278425,72.276588,98.503274,-26.226686,18.181818,16.666667,4.358065,2.035484,2.000000,3.803448,2.700000,3.351724,1.996552,3.871429,3.724138,3.342857,-0.003448,0.067980,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Los Angeles Angels
260,823481,2026-04-13,Final,Philadelphia Phillies,Chicago Cubs,143,112,13.0,7.0,Cristopher Sánchez,Javier Assad,650911,665871,L,R,99.434115,100.860716,25.263158,4.444444,1.846269,5.700000,1.0,True,2026,2026,20.818713,-3.853731,-1.426601,105.659495,105.061845,0.597650,18.750000,5.263158,1.745161,3.100000,2.332558,5.375000,4.186047,4.275000,1.818750,4.850000,3.937500,4.821429,-0.513808,-0.525000,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Chicago Cubs
261,823400,2026-04-13,Final,Pittsburgh Pirates,Washington Nationals,134,120,16.0,5.0,Paul Skenes,PJ Poulin,694973,676571,R,L,104.712540,110.133625,15.277778,0.000000,4.044444,7.700000,1.0,True,2026,2026,15.277778,-3.655556,-5.421085,97.610650,103.274088,-5.663438,16.279070,0.000000,3.629412,9.300000,4.258621,6.625547,3.910345,6.306569,3.610638,7.042857,3.734043,6.685714,-0.647982,0.417310,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Washington Nationals
262,823153,2026-04-13,Final,Seattle Mariners,Houston Astros,136,117,6.0,2.0,George Kirby,Mike Burrows,669923,681347,R,R,92.015788,114.270769,15.533981,10.679612,3.678313,5.509091,1.0,True,2026,2026,4.854369,-1.830778,-22.254981,98.362956,112.394761,-14.031805,13.461538,10.638298,4.171429,4.067742,2.822689,6.517722,2.042017,7.006329,2.215385,6.653398,1.038462,8.126214,-0.607304,0.135677,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Houston Astros
263,823073,2026-04-13,Final,St. Louis Cardinals,Cleveland Guardians,138,114,3.0,9.0,Matthew Liberatore,Gavin Williams,669461,668909,L,R,99.148795,99.434115,4.255319,14.606742,6.195238,4.379412,0.0,

In [12]:
# V6 model block (kept aligned in spirit with games_today_tomorrow)
s = favorites.copy()

# Stable scaling
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

# Tunable compression / penalty knobs
RISK_W_CONFLICT = 0.30
RISK_W_INSTABILITY = 0.40
RISK_W_SIGNAL_GAP = 0.30
RISK_TANH_ALPHA = 0.80
QUALITY_INSTABILITY_DECAY = 0.90
TRAP_PENALTY_WEIGHT = 0.35

OFF_W_OFFENSE = 0.85
OFF_W_PLATOON = 0.15
PEN_LEG_DAMP = 0.65
COHERENCE_SOFT_MIN = 0.50
COHERENCE_SOFT_RANGE = 0.50

# Missing SP/offense diffs: neutral 0 so norms stay finite
kbb = pd.to_numeric(s["sp_kbb_diff"], errors="coerce").fillna(0.0)
xfi = pd.to_numeric(s["sp_xfip_diff"], errors="coerce").fillna(0.0)
off = pd.to_numeric(s["offense_diff"], errors="coerce").fillna(0.0)
kbb_norm = kbb / SP_KBB_ABS
xfip_norm = -xfi / SP_XFIP_ABS
off_norm = off / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

# Pen: prefer roll14; neutralize missing values so matrix math is stable
if "home_pen_roll14_fip" in s.columns:
    hr_pen = s["home_pen_roll14_fip"]
    if "home_pen_season_fip" in s.columns:
        hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
else:
    hr_pen = pd.Series(np.nan, index=s.index)

if "away_pen_roll14_fip" in s.columns:
    ar_pen = s["away_pen_roll14_fip"]
    if "away_pen_season_fip" in s.columns:
        ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
else:
    ar_pen = pd.Series(np.nan, index=s.index)

pen_gap = hr_pen - ar_pen  # home - away; lower home FIP better
pen_norm = (-pen_gap / PEN_ABS).fillna(0.0)

sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
off_vector = (OFF_W_OFFENSE * off_norm) + (OFF_W_PLATOON * platoon_norm)
pen_vector = PEN_LEG_DAMP * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

agreement = 1 - np.nanvar(sign_matrix, axis=0)
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
coherence_mult = COHERENCE_SOFT_MIN + COHERENCE_SOFT_RANGE * coherence_score
raw_edge = mean_mag
instability = np.std(signal_matrix, axis=0)

direction_conflict = (
    (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
    + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
    + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
)

risk_penalty_raw = (
    RISK_W_CONFLICT * direction_conflict
    + RISK_W_INSTABILITY * instability
    + RISK_W_SIGNAL_GAP * np.abs(sp_vector - off_vector)
)
risk_penalty = np.tanh(RISK_TANH_ALPHA * risk_penalty_raw)

trap_score = raw_edge * (1 - coherence_score)
quality_score = raw_edge * coherence_mult * np.exp(-QUALITY_INSTABILITY_DECAY * instability) * (1 / (1 + risk_penalty))
risk_adjusted_edge = quality_score - TRAP_PENALTY_WEIGHT * trap_score

prefer_home = sp_vector >= 0
s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = direction_conflict
s["instability"] = instability
s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score
s["risk_penalty_raw"] = risk_penalty_raw
s["risk_penalty"] = risk_penalty

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int)
    + (np.abs(xfip_norm) > 1).astype(int)
    + (np.abs(off_norm) > 1).astype(int)
)

# Backtest outcome: did the model's favored side win?
if "home_win" in s.columns:
    hw = pd.to_numeric(s["home_win"], errors="coerce")
    s["favorite_won"] = np.where(prefer_home, hw == 1.0, hw == 0.0)
else:
    s["favorite_won"] = np.nan

# Kalshi is optional in historical backtests.
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(s.columns):
    s["implied_prob"] = np.where(prefer_home, s["kalshi_home_implied"], s["kalshi_away_implied"])
    s["market_edge"] = s["risk_adjusted_edge"] - s["implied_prob"]
else:
    s["implied_prob"] = np.nan
    s["market_edge"] = np.nan

scored = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False]).reset_index(drop=True)

show_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side", "favorite_won",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "signal_agreement", "signal_conflict", "core_matches",
    "sp_kbb_diff", "sp_xfip_diff", "offense_diff", "offense_platoon_diff",
    "implied_prob", "market_edge", "home_win",
]
show_cols = [c for c in show_cols if c in scored.columns]

print(f"Scored games: {len(scored)}")
display(scored[show_cols].head(25))

# Optional aggregate view (uncomment to use):
# display(
#     scored["favorite_won"]
#     .map({True: "Favorite Won", False: "Favorite Did Not Win"})
#     .fillna("Unknown")
#     .value_counts(dropna=False)
#     .rename_axis("favorite_result")
#     .to_frame("count")
# )

Scored games: 282


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,favorite_won,risk_adjusted_edge,quality_score,coherence_score,instability,signal_agreement,signal_conflict,core_matches,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,implied_prob,market_edge,home_win
0,2026-04-01,Los Angeles Dodgers,Cleveland Guardians,Los Angeles Dodgers,home_favored,False,0.770028,0.961196,0.700675,0.167407,3,0,3,4.340627,-0.928762,20.115079,18.802619,NaN,NaN,0.0
1,2026-04-01,Atlanta Braves,Athletics,Atlanta Braves,home_favored,True,0.716411,0.943095,0.685282,0.257816,3,0,2,8.675203,0.246082,18.545818,28.274614,NaN,NaN,1.0
2,2026-04-07,Los Angeles Angels,Atlanta Braves,Atlanta Braves,away_favored,True,0.371321,0.510461,0.669662,0.365097,0,3,0,-2.982879,0.367384,-8.131628,-14.009766,NaN,NaN,0.0
3,2026-04-07,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.348804,0.437335,0.698666,0.178463,3,0,1,2.580645,-0.552582,6.134386,5.287003,NaN,NaN,1.0
4,2026-04-06,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.336708,0.443822,0.683795,0.267295,3,0,1,1.922121,-0.828125,6.134386,6.594949,NaN,NaN,1.0
5,2026-03-31,Atlanta Braves,Athletics,Atlanta Braves,home_favored,False,0.325551,0.708299,0.631606,0.723174,3,0,3,5.782848,-2.285106,18.545818,28.274614,NaN,NaN,0.0
6,2026-04-08,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.278223,0.337067,0.708442,0.126545,3,0,2,4.819005,0.896893,6.134386,5.287003,NaN,NaN,1.0
7,2026-04-09,Miami Marlins,Cincinnati Reds,Miami Marlins,home_favored,True,0.261737,0.411547,0.658284,0.455523,3,0,2,5.253623,0.109997,15.264635,27.081384,NaN,NaN,1.0
8,2026-04-12,Atlanta Braves,Cleveland Guardians,Atlanta Braves,home_favored,True,0.235918,0.413921,0.639070,0.638858,3,0,2,3.343382,-0.263636,11.983451,5.622982,NaN,NaN,1.0
9,2026-03-30,Seattle Mariners,New York Yankees,New York Yankees,away_favored,False,0.220378,0.434783,0.634217,0.692707,0,3,2,-7.204301,-0.809524,-5.991726,-32.840071,NaN,NaN,1.0


In [13]:
# Additional breakdowns by metric bands
band_df = scored.copy()

# Keep known outcomes for cleaner win-rate summaries
band_df = band_df[band_df["favorite_won"].isin([True, False])].copy()

band_specs = {
    "risk_adjusted_edge": [-np.inf, 0.25, 0.50, 0.75, 1.00, np.inf],
    "quality_score": [-np.inf, 0.10, 0.20, 0.30, 0.40, np.inf],
    "coherence_score": [-np.inf, 0.40, 0.50, 0.60, 0.70, np.inf],
    "instability": [-np.inf, 0.80, 1.00, 1.20, 1.50, np.inf],
}

for metric, bins in band_specs.items():
    if metric not in band_df.columns:
        continue

    d = band_df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        continue

    d["band"] = pd.cut(d[metric], bins=bins, include_lowest=True)

    out = (
        d.groupby("band", observed=False)["favorite_won"]
        .agg(
            games="count",
            favorite_wins=lambda x: int((x == True).sum()),
            favorite_losses=lambda x: int((x == False).sum()),
            favorite_win_rate="mean",
        )
        .reset_index()
    )
    out["favorite_win_rate"] = out["favorite_win_rate"].round(3)

    print(f"\n{metric} band breakdown")
    display(out)



risk_adjusted_edge band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.25]",274,150,124,0.547
1,"(0.25, 0.5]",6,5,1,0.833
2,"(0.5, 0.75]",1,1,0,1.000
3,"(0.75, 1.0]",1,0,1,0.000
4,"(1.0, inf]",0,0,0,NaN



quality_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.1]",183,106,77,0.579
1,"(0.1, 0.2]",50,25,25,0.500
2,"(0.2, 0.3]",30,13,17,0.433
3,"(0.3, 0.4]",6,3,3,0.500
4,"(0.4, inf]",13,9,4,0.692



coherence_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.4]",153,87,66,0.569
1,"(0.4, 0.5]",35,19,16,0.543
2,"(0.5, 0.6]",57,29,28,0.509
3,"(0.6, 0.7]",35,20,15,0.571
4,"(0.7, inf]",2,1,1,0.500



instability band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.8]",36,19,17,0.528
1,"(0.8, 1.0]",20,11,9,0.550
2,"(1.0, 1.2]",20,12,8,0.600
3,"(1.2, 1.5]",29,18,11,0.621
4,"(1.5, inf]",177,96,81,0.542


In [14]:
# Decision / confidence layers + backtest summary
def decision_layer(df: pd.DataFrame) -> pd.Series:
    # Relaxed vs 1.0/0.60/1.2 and 0.55/0.45 — avoids starving BET count
    conditions = [
        (df["risk_adjusted_edge"] > 0.72) & (df["coherence_score"] > 0.52) & (df["instability"] < 1.35),
        (df["risk_adjusted_edge"] > 0.48) & (df["coherence_score"] > 0.40),
    ]
    choices = ["BET", "LEAN"]
    return np.select(conditions, choices, default="PASS")


def confidence_tier(df: pd.DataFrame) -> pd.Series:
    return np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.68),
        "A (Strong Bet)",
        np.where(
            df["decision"] == "BET",
            "B (Playable Bet)",
            np.where(df["decision"] == "LEAN", "C (Lean Only)", "D (Pass)"),
        ),
    )


bt = scored.copy()
bt["decision"] = decision_layer(bt)
bt["tier"] = confidence_tier(bt)

# Evaluate pick correctness only on non-PASS rows
bt_eval = bt[bt["decision"].isin(["BET", "LEAN"])].copy()
bt_eval["pick_home"] = bt_eval["_mismatch_side"].eq("home_favored")
bt_eval["won"] = np.where(bt_eval["pick_home"], bt_eval["home_win"] == 1.0, bt_eval["home_win"] == 0.0)

summary = {
    "rows_scored": int(len(bt)),
    "rows_bet_or_lean": int(len(bt_eval)),
    "win_rate_bet_or_lean": float(bt_eval["won"].mean()) if len(bt_eval) else np.nan,
    "bets_only": int((bt["decision"] == "BET").sum()),
    "leans_only": int((bt["decision"] == "LEAN").sum()),
}

print(summary)
if len(bt_eval):
    by_tier = bt_eval.groupby("tier")["won"].agg(["count", "mean"]).rename(columns={"mean": "win_rate"})
    display(by_tier.sort_values("count", ascending=False))

view_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "decision", "tier",
    "risk_adjusted_edge", "coherence_score", "instability", "home_win", "won",
]
view_cols = [c for c in view_cols if c in bt_eval.columns]
display(bt_eval.sort_values("risk_adjusted_edge", ascending=False)[view_cols].head(50))

# Slate Top-N evaluation (default Top-3 per date) on ALL scored games with known outcomes
TOP_N = 3
if "favorite_won" in scored.columns:
    top_pool = scored[scored["favorite_won"].isin([True, False])].copy()
    top_pool["won"] = top_pool["favorite_won"].astype(bool)
    if len(top_pool):
        bt_topn = (
            top_pool.sort_values(["game_date", "risk_adjusted_edge"], ascending=[True, False])
            .groupby("game_date", as_index=False, group_keys=False)
            .head(TOP_N)
            .copy()
        )
        topn_summary = {
            "top_n": TOP_N,
            "slates": int(bt_topn["game_date"].nunique()),
            "topn_rows": int(len(bt_topn)),
            "topn_win_rate": float(bt_topn["won"].mean()) if len(bt_topn) else np.nan,
        }
        print("Top-N slate summary:", topn_summary)
        by_date = bt_topn.groupby("game_date")["won"].agg(picks="count", wins="sum", win_rate="mean").reset_index()
        display(by_date.head(20))
        top_cols = [c for c in view_cols if c in bt_topn.columns] + [c for c in ["favorite_won"] if c in bt_topn.columns]
        display(bt_topn.sort_values(["game_date", "risk_adjusted_edge"], ascending=[False, False])[top_cols].head(50))

{'rows_scored': 282, 'rows_bet_or_lean': 2, 'win_rate_bet_or_lean': 0.5, 'bets_only': 0, 'leans_only': 2}


,count,win_rate
tier,,
C (Lean Only),2,0.5


,game_date,home_team_name,away_team_name,favored_team,decision,tier,risk_adjusted_edge,coherence_score,instability,home_win,won
0,2026-04-01,Los Angeles Dodgers,Cleveland Guardians,Los Angeles Dodgers,LEAN,C (Lean Only),0.770028,0.700675,0.167407,0.0,False
1,2026-04-01,Atlanta Braves,Athletics,Atlanta Braves,LEAN,C (Lean Only),0.716411,0.685282,0.257816,1.0,True


Top-N slate summary: {'top_n': 3, 'slates': 22, 'topn_rows': 64, 'topn_win_rate': 0.640625}


,game_date,picks,wins,win_rate
0,2026-03-25,1,1,1.000000
1,2026-03-26,3,1,0.333333
2,2026-03-27,3,3,1.000000
3,2026-03-28,3,3,1.000000
4,2026-03-29,3,2,0.666667
5,2026-03-30,3,0,0.000000
6,2026-03-31,3,2,0.666667
7,2026-04-01,3,1,0.333333
8,2026-04-02,3,3,1.000000
9,2026-04-03,3,1,0.333333


,game_date,home_team_name,away_team_name,favored_team,risk_adjusted_edge,coherence_score,instability,home_win,won,favorite_won
24,2026-04-15,Baltimore Orioles,Arizona Diamondbacks,Baltimore Orioles,0.089408,0.624355,0.813829,0.0,False,False
135,2026-04-15,St. Louis Cardinals,Cleveland Guardians,St. Louis Cardinals,-0.235482,0.381268,1.638717,1.0,True,True
143,2026-04-15,Minnesota Twins,Boston Red Sox,Boston Red Sox,-0.255413,0.382498,1.604885,0.0,True,True
14,2026-04-14,Detroit Tigers,Kansas City Royals,Detroit Tigers,0.164090,0.639789,0.631168,1.0,True,True
22,2026-04-14,Athletics,Texas Rangers,Texas Rangers,0.113666,0.649239,0.536438,1.0,False,False
40,2026-04-14,Cincinnati Reds,San Francisco Giants,San Francisco Giants,0.003213,0.466467,0.389338,1.0,False,False
11,2026-04-13,Atlanta Braves,Miami Marlins,Atlanta Braves,0.198768,0.648654,0.541978,0.0,False,False
27,2026-04-13,Athletics,Texas Rangers,Texas Rangers,0.062769,0.623026,0.831492,0.0,True,True
47,2026-04-13,Minnesota Twins,Boston Red Sox,Boston Red Sox,-0.041630,0.435480,0.678358,1.0,False,False
8,2026-04-12,Atlanta Braves,Cleveland Guardians,Atlanta Braves,0.235918,0.639070,0.638858,1.0,True,True


In [15]:
# Quantile calibration tables + monotonicity checks

cal_df = scored.copy()
if "favorite_won" not in cal_df.columns:
    raise ValueError("favorite_won missing; run V6 scoring cell first.")

cal_df = cal_df[cal_df["favorite_won"].isin([True, False])].copy()
if cal_df.empty:
    raise ValueError("No rows with known favorite_won values for calibration.")

metrics = [
    "risk_adjusted_edge",
    "quality_score",
    "coherence_score",
    "instability",
]


def quantile_calibration_table(df: pd.DataFrame, metric: str, q: int = 10, ascending_good: bool = True):
    d = df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        return None, None

    # Handle low-variance metrics safely.
    n_unique = d[metric].nunique(dropna=True)
    q_eff = max(2, min(q, int(n_unique)))

    d["quantile"] = pd.qcut(d[metric], q=q_eff, duplicates="drop")
    tab = (
        d.groupby("quantile", observed=False)["favorite_won"]
        .agg(games="count", favorite_wins="sum", favorite_win_rate="mean")
        .reset_index()
    )
    tab["favorite_losses"] = tab["games"] - tab["favorite_wins"]
    tab["favorite_win_rate"] = tab["favorite_win_rate"].round(3)

    rates = tab["favorite_win_rate"].to_numpy()
    if not ascending_good:
        rates = rates[::-1]
    deltas = np.diff(rates)
    mono_score = float((deltas >= 0).mean()) if len(deltas) else np.nan
    is_monotonic = bool((deltas >= 0).all()) if len(deltas) else True

    summary = {
        "metric": metric,
        "quantile_bins": int(len(tab)),
        "rows_used": int(tab["games"].sum()),
        "win_rate_first_bin": float(tab.iloc[0]["favorite_win_rate"]),
        "win_rate_last_bin": float(tab.iloc[-1]["favorite_win_rate"]),
        "lift_last_minus_first": float(tab.iloc[-1]["favorite_win_rate"] - tab.iloc[0]["favorite_win_rate"]),
        "is_monotonic_expected_direction": is_monotonic,
        "monotonicity_fraction": round(mono_score, 3) if not np.isnan(mono_score) else np.nan,
    }
    return tab, summary


# For instability, lower values are generally better; reverse expected direction.
expected_ascending = {
    "risk_adjusted_edge": True,
    "quality_score": True,
    "coherence_score": True,
    "instability": False,
}

summaries = []
for m in metrics:
    if m not in cal_df.columns:
        continue
    tab, s = quantile_calibration_table(cal_df, m, q=10, ascending_good=expected_ascending[m])
    if tab is None:
        continue
    print(f"\n{m}: quantile calibration")
    display(tab)
    summaries.append(s)

if summaries:
    print("\nMonotonicity summary")
    display(pd.DataFrame(summaries).sort_values("metric").reset_index(drop=True))


risk_adjusted_edge: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-1.63, -0.69]",29,19,0.655,10
1,"(-0.69, -0.541]",28,17,0.607,11
2,"(-0.541, -0.415]",28,13,0.464,15
3,"(-0.415, -0.357]",29,13,0.448,16
4,"(-0.357, -0.248]",27,20,0.741,7
5,"(-0.248, -0.204]",28,14,0.500,14
6,"(-0.204, -0.129]",34,17,0.500,17
7,"(-0.129, -0.0761]",22,11,0.500,11
8,"(-0.0761, 0.0592]",28,16,0.571,12
9,"(0.0592, 0.77]",29,16,0.552,13



quality_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.00213, 0.023]",29,18,0.621,11
1,"(0.023, 0.0392]",28,15,0.536,13
2,"(0.0392, 0.0511]",28,11,0.393,17
3,"(0.0511, 0.0655]",28,18,0.643,10
4,"(0.0655, 0.0733]",28,18,0.643,10
5,"(0.0733, 0.085]",28,18,0.643,10
6,"(0.085, 0.116]",28,15,0.536,13
7,"(0.116, 0.171]",28,13,0.464,15
8,"(0.171, 0.277]",34,15,0.441,19
9,"(0.277, 0.961]",23,15,0.652,8



coherence_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.228, 0.341]",29,17,0.586,12
1,"(0.341, 0.355]",28,17,0.607,11
2,"(0.355, 0.364]",28,13,0.464,15
3,"(0.364, 0.375]",28,12,0.429,16
4,"(0.375, 0.389]",28,21,0.750,7
5,"(0.389, 0.416]",28,15,0.536,13
6,"(0.416, 0.559]",28,19,0.679,9
7,"(0.559, 0.584]",31,11,0.355,20
8,"(0.584, 0.617]",25,15,0.600,10
9,"(0.617, 0.708]",29,16,0.552,13



instability: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.10099999999999999, 0.712]",29,16,0.552,13
1,"(0.712, 1.009]",28,15,0.536,13
2,"(1.009, 1.341]",28,17,0.607,11
3,"(1.341, 1.566]",28,18,0.643,10
4,"(1.566, 1.668]",28,8,0.286,20
5,"(1.668, 2.029]",28,20,0.714,8
6,"(2.029, 2.32]",28,10,0.357,18
7,"(2.32, 2.697]",28,16,0.571,12
8,"(2.697, 3.412]",28,17,0.607,11
9,"(3.412, 6.155]",29,19,0.655,10



Monotonicity summary


,metric,quantile_bins,rows_used,win_rate_first_bin,win_rate_last_bin,lift_last_minus_first,is_monotonic_expected_direction,monotonicity_fraction
0,coherence_score,10,282,0.586,0.552,-0.034,False,0.444
1,instability,10,282,0.552,0.655,0.103,False,0.333
2,quality_score,10,282,0.621,0.652,0.031,False,0.444
3,risk_adjusted_edge,10,282,0.655,0.552,-0.103,False,0.444


In [16]:
# Post-model calibration (logistic + isotonic), quantile diagnostics, and YoY-style summary schema
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

work = scored.copy()
work = work[work["favorite_won"].isin([True, False])].copy()
if work.empty:
    raise ValueError("No rows with favorite_won for calibration.")

work["y"] = work["favorite_won"].astype(int)
work["score_raw"] = pd.to_numeric(work["risk_adjusted_edge"], errors="coerce")
work = work.dropna(subset=["score_raw", "y"]).copy()

# Time-aware split by game_date to reduce leakage.
unique_dates = sorted(pd.to_datetime(work["game_date"]).dropna().unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough unique dates for calibration split.")
cut_idx = max(1, int(len(unique_dates) * 0.7))
train_dates = set(unique_dates[:cut_idx])
valid_dates = set(unique_dates[cut_idx:])

train = work[work["game_date"].isin(train_dates)].copy()
valid = work[work["game_date"].isin(valid_dates)].copy()
if train.empty or valid.empty:
    raise ValueError("Train/valid split empty; adjust date window.")

xtr = train[["score_raw"]].to_numpy()
ytr = train["y"].to_numpy()
xva = valid[["score_raw"]].to_numpy()
yva = valid["y"].to_numpy()

# Baseline probability proxy from score scaling
smin, smax = float(train["score_raw"].min()), float(train["score_raw"].max())
den = (smax - smin) if (smax - smin) > 1e-12 else 1.0
train["p_raw"] = ((train["score_raw"] - smin) / den).clip(0, 1)
valid["p_raw"] = ((valid["score_raw"] - smin) / den).clip(0, 1)

# Logistic (Platt-style)
logit = LogisticRegression(max_iter=2000)
logit.fit(xtr, ytr)
train["p_model_logit"] = logit.predict_proba(xtr)[:, 1]
valid["p_model_logit"] = logit.predict_proba(xva)[:, 1]

# Isotonic on raw score
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(train["score_raw"].to_numpy(), ytr)
train["p_model_iso"] = iso.predict(train["score_raw"].to_numpy())
valid["p_model_iso"] = iso.predict(valid["score_raw"].to_numpy())

# Write calibrated probabilities back to scored
scored = scored.copy()
for col in ["p_model_logit", "p_model_iso"]:
    scored[col] = np.nan
scored.loc[train.index, "p_model_logit"] = train["p_model_logit"]
scored.loc[valid.index, "p_model_logit"] = valid["p_model_logit"]
scored.loc[train.index, "p_model_iso"] = train["p_model_iso"]
scored.loc[valid.index, "p_model_iso"] = valid["p_model_iso"]


def metric_row(name: str, y_true: np.ndarray, p: np.ndarray) -> dict:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        "model": name,
        "brier": float(brier_score_loss(y_true, p)),
        "logloss": float(log_loss(y_true, p)),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else np.nan,
    }

metric_rows = [
    metric_row("raw_scaled", yva, valid["p_raw"].to_numpy()),
    metric_row("logit", yva, valid["p_model_logit"].to_numpy()),
    metric_row("isotonic", yva, valid["p_model_iso"].to_numpy()),
]
metrics_df = pd.DataFrame(metric_rows).sort_values("brier")
print("Validation calibration metrics")
display(metrics_df)


def quantile_hit_rate(df: pd.DataFrame, score_col: str, y_col: str = "y", q: int = 10):
    d = df[[score_col, y_col]].dropna().copy()
    if d.empty or d[score_col].nunique() < 2:
        return pd.DataFrame(), {"metric": score_col, "monotonicity_fraction": np.nan, "lift_last_minus_first": np.nan}
    q_eff = min(q, int(d[score_col].nunique()))
    d["quantile"] = pd.qcut(d[score_col], q=q_eff, duplicates="drop")
    tab = d.groupby("quantile", observed=False)[y_col].agg(games="count", win_rate="mean").reset_index()
    tab["win_rate"] = tab["win_rate"].round(3)
    deltas = np.diff(tab["win_rate"].to_numpy())
    mono = float((deltas >= 0).mean()) if len(deltas) else np.nan
    lift = float(tab.iloc[-1]["win_rate"] - tab.iloc[0]["win_rate"]) if len(tab) else np.nan
    return tab, {"metric": score_col, "monotonicity_fraction": round(mono, 3) if not np.isnan(mono) else np.nan, "lift_last_minus_first": lift}

quant_targets = ["score_raw", "quality_score", "coherence_score", "p_model_logit", "p_model_iso"]
mono_rows = []
for col in quant_targets:
    if col not in valid.columns:
        continue
    qt, ms = quantile_hit_rate(valid, col, y_col="y", q=10)
    if len(qt):
        print(f"\nValidation quantile hit-rate: {col}")
        display(qt)
    mono_rows.append(ms)

mono_df = pd.DataFrame(mono_rows)
print("\nMonotonicity summary")
display(mono_df)

# Standardized summary schema for cross-year comparison
summary_row = {
    "season": int(SEASON),
    "rows_scored": int(len(scored)),
    "rows_with_outcome": int(len(work)),
    "train_rows": int(len(train)),
    "valid_rows": int(len(valid)),
    "top_n_default": 3,
}

# Include top-N result if prior cell produced it
if "bt_topn" in globals() and isinstance(bt_topn, pd.DataFrame) and len(bt_topn):
    summary_row["topn_rows"] = int(len(bt_topn))
    summary_row["topn_win_rate"] = float(bt_topn["won"].mean())
else:
    summary_row["topn_rows"] = np.nan
    summary_row["topn_win_rate"] = np.nan

best_cal = metrics_df.iloc[0]
summary_row["best_model_by_brier"] = best_cal["model"]
summary_row["best_brier"] = float(best_cal["brier"])
summary_row["best_logloss"] = float(best_cal["logloss"])
summary_row["best_auc"] = float(best_cal["auc"]) if pd.notna(best_cal["auc"]) else np.nan

year_summary = pd.DataFrame([summary_row])
print("\nYear summary row")
display(year_summary)

Validation calibration metrics


,model,brier,logloss,auc
1,logit,0.244307,0.681713,0.509309
2,isotonic,0.244494,0.682119,0.500000
0,raw_scaled,0.261851,0.720587,0.490691



Validation quantile hit-rate: score_raw


,quantile,games,win_rate
0,"(-1.168, -0.642]",8,0.750
1,"(-0.642, -0.564]",8,0.375
2,"(-0.564, -0.435]",8,0.500
3,"(-0.435, -0.351]",8,0.625
4,"(-0.351, -0.255]",8,0.875
5,"(-0.255, -0.179]",7,0.714
6,"(-0.179, -0.104]",8,0.500
7,"(-0.104, -0.0392]",8,0.375
8,"(-0.0392, 0.118]",8,0.500
9,"(0.118, 0.262]",8,0.750



Validation quantile hit-rate: quality_score


...

,quantile,games,win_rate
0,"(0.503, 0.512]",8,0.750
1,"(0.512, 0.522]",8,0.500
2,"(0.522, 0.525]",8,0.375
3,"(0.525, 0.53]",8,0.500
4,"(0.53, 0.534]",8,0.750
5,"(0.534, 0.54]",7,0.857
6,"(0.54, 0.545]",8,0.625
7,"(0.545, 0.552]",8,0.500
8,"(0.552, 0.557]",8,0.375
9,"(0.557, 0.587]",8,0.750



Monotonicity summary


,metric,monotonicity_fraction,lift_last_minus_first
0,score_raw,0.556,0.000
1,quality_score,0.444,-0.125
2,coherence_score,0.333,-0.250
3,p_model_logit,0.444,0.000
4,p_model_iso,NaN,NaN



Year summary row


,season,rows_scored,rows_with_outcome,train_rows,valid_rows,top_n_default,topn_rows,topn_win_rate,best_model_by_brier,best_brier,best_logloss,best_auc
0,2026,282,282,203,79,3,64,0.640625,logit,0.244307,0.681713,0.509309


## Metric definitions and acceptance bars (winner prediction)

- **Brier score**: mean squared error of predicted probability vs actual outcome (`0/1`). Lower is better.
- **Log loss**: penalizes confident wrong probabilities more than Brier. Lower is better.
- **AUC**: probability that a random win is ranked above a random loss. `0.5` is random, `1.0` is perfect.
- **Monotonicity fraction**: fraction of adjacent quantile bins where win rate moves in expected direction.
- **Lift (last-first)**: difference in win rate between highest and lowest quantile bins.
- **Top-N win rate**: win rate among the top-N ranked picks per slate date.

Suggested minimum bars for practical actionability (outright winners):

- `AUC >= 0.53`
- `best_brier <= 0.245`
- `monotonicity_fraction >= 0.60` on calibrated probability
- `lift_last_minus_first >= 0.03` on calibrated probability
- `topn_win_rate >= 0.54` for `TOP_N=3`


In [17]:
# Acceptance-bar check for actionability (winner prediction)
# Requires year_summary, metrics_df, mono_df from the calibration cell.

if "year_summary" not in globals() or "metrics_df" not in globals() or "mono_df" not in globals():
    raise ValueError("Run calibration/summary cell first.")

bars = {
    "min_auc": 0.53,
    "max_brier": 0.245,
    "min_mono": 0.60,
    "min_lift": 0.03,
    "min_topn_win_rate": 0.54,
}

ys = year_summary.iloc[0]
prob_row = mono_df[mono_df["metric"].isin(["p_model_logit", "p_model_iso"])].copy()
if prob_row.empty:
    mono_best = np.nan
    lift_best = np.nan
else:
    mono_best = float(prob_row["monotonicity_fraction"].max())
    lift_best = float(prob_row["lift_last_minus_first"].max())

checks = pd.DataFrame(
    [
        {"check": "AUC", "value": float(ys.get("best_auc", np.nan)), "bar": bars["min_auc"], "pass": float(ys.get("best_auc", np.nan)) >= bars["min_auc"]},
        {"check": "Brier", "value": float(ys.get("best_brier", np.nan)), "bar": bars["max_brier"], "pass": float(ys.get("best_brier", np.nan)) <= bars["max_brier"]},
        {"check": "Monotonicity (prob)", "value": mono_best, "bar": bars["min_mono"], "pass": mono_best >= bars["min_mono"] if pd.notna(mono_best) else False},
        {"check": "Lift (prob)", "value": lift_best, "bar": bars["min_lift"], "pass": lift_best >= bars["min_lift"] if pd.notna(lift_best) else False},
        {"check": "Top-3 win rate", "value": float(ys.get("topn_win_rate", np.nan)), "bar": bars["min_topn_win_rate"], "pass": float(ys.get("topn_win_rate", np.nan)) >= bars["min_topn_win_rate"]},
    ]
)

print("Actionability checks")
display(checks)
print("Passed", int(checks["pass"].sum()), "of", len(checks), "checks")

Actionability checks


,check,value,bar,pass
0,AUC,0.509309,0.530,False
1,Brier,0.244307,0.245,True
2,Monotonicity (prob),0.444000,0.600,False
3,Lift (prob),0.000000,0.030,False
4,Top-3 win rate,0.640625,0.540,True


Passed 2 of 5 checks
